# Offline Diffusion DQN

Copy of example 2 with a diffusion head conditioned on the backbone's final
readout features. The environment, tokenizer, encoder, and data pipeline are unchanged.
We extend the Q-only flow-matching loss of
[Value Forcing, Appendix D.5](https://arxiv.org/html/2611.02160v1#A4.SS5).
The action values evolve jointly from independent Gaussian coordinates.
One Bellman backup reaches the next decision; every diffusion integration step
contributes a velocity loss. There are no n-step or multi-step horizon options.


In [ ]:
import json
from copy import deepcopy

import torch
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    Augmenter,
    DataLoader,
    Tokenizer,
    compose,
    load_stores_from_hub,
    save_tokenizer,
)
from mouse_core.models import (
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    prepare_decode,
    save_model,
)
from mouse_core.models.mlp import SwiGLU
from mouse_core.objectives import DiffusionDqnObjective, boundary_discount

# Repositories
# Hugging Face dataset repo for load_stores_from_hub
DATASET_ID = "mouse-example-frozenlake-dataset"
# Local checkpoint directory
OUTPUT = "runs/offline-diffusion-dqn"

# Environment
# number of discrete actions predicted by the head
MAX_ACTIONS = 4
# vocabulary size for discrete observations
MAX_OBS_DISCRETE = 64
# max steps per episode (matches collect)
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task (matches collect)
REPEAT_COUNT = 20

# Model
# HF checkpoint for Tokenizer, Embedding, and Backbone
PRETRAINED = "Qwen/Qwen3-0.6B"
DIFFUSION_NUM_BLOCKS = 2
# Applied once to the final velocity linear layer
VELOCITY_OUTPUT_INITIALIZATION_SCALE = 1.0
# Sinusoidal basis has twice this many components before its learned projection
DIFFUSION_TIME_FREQUENCIES = 64
DIFFUSION_TIME_MIN = 0.0
DIFFUSION_TIME_MAX = 1.0
# Sine/cosine pairs per action; one shared reference range
DIFFUSION_LATENT_FREQUENCIES = 64
DIFFUSION_LATENT_MIN = 0.0
DIFFUSION_LATENT_MAX = 20.0
# Fixed multipliers on the three input embeddings, before summation
DIFFUSION_CONTEXT_EMBEDDING_SCALE = 1.0
DIFFUSION_LATENT_EMBEDDING_SCALE = 1.0
DIFFUSION_TIME_EMBEDDING_SCALE = 1.0
DIFFUSION_STEPS = 10
# Independent full trajectories averaged to choose the training backup action
DIFFUSION_ACTION_SAMPLES = 1
# Gaussian starting-noise mean and standard deviation
DIFFUSION_NOISE_MEAN = 0.0
DIFFUSION_NOISE_STD = 1.0

# Compute text before each value readout
COMPUTE_TOKENS = 8

# Training
# packed tokens; whole episodes are added until the next one would not fit
TOKEN_BUDGET = 16384
# outer train cycles (print cadence)
NUM_CYCLES = 2
# optimizer updates per cycle (passed to run_train)
TRAIN_STEPS = 50

# Optimizer learning rates
# token embeddings and numeric Fourier features
LR_EMBEDDING = 1e-05
# backbone
LR_BACKBONE = 1e-05
# prediction heads
LR_HEADS = 1e-05

# Target interpolation rates
# delayed embedding
POLYAK_TAU_EMBEDDING = 0.1
# delayed backbone
POLYAK_TAU_BACKBONE = 0.1
# Delayed velocity-network rate
POLYAK_TAU_VALUE = 0.001

# Runtime
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved
`Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)

## Data Pipeline

`DataLoader` samples complete outer tasks with
`sample_ranges=sample_ranges`, keeping all repeated lake episodes together.
Each sampled task gets a unique sequence ID so attention and objective transitions
cannot cross between tasks.

Pass exactly one of `token_budget` or `events_budget`. Each batch contains whole samples
up to that budget. An oversized first sample raises. Token budgets count packed tokens,
including episode prompts; event budgets count stored source events, not decision
readouts.

The pipeline is
`source events → sparse decisions → augmenter → tokenizer → pack → embedding`:

| Stage       | Role                                                                                                             |
| ----------- | ---------------------------------------------------------------------------------------------------------------- |
| `Augmenter` | Permutes actions and observations consistently within a sampled task; action permutations also update `_q_star`. |
| `Tokenizer` | Tokenizes matching fields per event, groups tokens by `step_index`, and retains explicit objective labels.       |

`train_transform = compose(stages=(augmenter.events, tokenizer.encode))` is passed as
the loader’s `transform`. `next_batch()` returns `(inputs, objective_data)` as
TensorDicts; pass `objective_data["sequence_ids"]` as the objective’s `sequence_ids`.
[Inference](15_inference.ipynb) uses the saved tokenizer without augmentation so
selected actions match the environment.

For per-event encoding, configure `Tokenizer` with a flat `input_fields` list,
`step_field="step_index"`, and `episode_field="episode_index"`, then pass
`transform=tokenizer.encode` directly to the loader. It accumulates event tokens until
the step marker changes. Define a local range function to select windows of whole steps
N whole steps within an episode. Augmentation for this path must transform event
payloads before encoding; `augmenter.events` preserves source events and shares
episode-keyed draws.

`COMPUTE_TOKENS` sets a constant number of compute tokens before every value readout.
Set it to zero to disable compute text. The field filters on `action_requested`
and uses the literal format `" ." * COMPUTE_TOKENS`. No callback or runtime context
is needed. The configured text is saved with the tokenizer and reused at inference.

Qwen encodes each spaced period as one token (ID 659); ten bare periods can merge
into just one token. Passive and
terminal events emit no compute text. Compute tokens count toward the batch budget.

Only the newline has `head_output=True`. The diffusion loss trains the compute
positions through causal attention.


In [ ]:
def action_name(action):
    return ("a", "b", "c", "d")[action]


episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\nx marks no action at the beginning of an episode.\n"
)

augmenter = Augmenter(
    seed_field="episode_index",
    fields=[
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "action",
            "input_vector_field": "_q_star",
            "vocab_size": MAX_ACTIONS,
        },
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
        },
    ],
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": 'episode {repeat_episode_index+1}\n',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "x",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "format": '{action_name(action)}',
            "format_functions": {"action_name": action_name},
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ',{observation}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ',r={reward:g}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": " ." * COMPUTE_TOKENS,
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": ["inner_terminated", "inner_truncated"],
            "format": "\n\n",
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
    ],
)


train_transform = compose(stages=(augmenter.events, tokenizer.encode))


def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Train only on episodes with a recorded final environment result."""
    last = events[-1]
    return last["provider"] == "environment" and (
        last["payload"].get("_terminated") is True
        or last["payload"].get("_truncated") is True
    )


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    transform=train_transform,
    prefetch=4,
    num_workers=4,
)

## Build Encoder and Diffusion Head

The encoder is copied verbatim from example 2. `DiffusionHead` conditions on its
final features `C`, the complete noisy action-value vector `Z`, and flow time.
The velocity network automatically uses the backbone feature dimension `D`.
Context passes directly through RMSNorm (`eps=1e-5`, matching the DQN head)
without affine parameters or a learned projection: `e_C = RMSNorm_C(C)`.

The latent and time encodings use explicit reference bounds. For either scalar
input `x`, normalize `u = (x - min)/(max - min)`, then form `K` sine/cosine pairs:
`phi(x) = concat(sin(pi*(k/K)*u), cos(pi*(k/K)*u))`, for `k=1,...,K`.
The highest-frequency phase goes from zero to `pi` across the reference range;
none of the pairs completes a cycle within that range. Bounds must be finite
with a finite positive width. Values outside the range are not clipped.

Each action's latent value uses `DIFFUSION_LATENT_FREQUENCIES = K_Z` pairs and the
same `DIFFUSION_LATENT_MIN`/`DIFFUSION_LATENT_MAX` bounds, defaulting to `0`/`20`.
Concatenate the action encodings, then project once:
`e_Z = W_Z concat_a(phi(Z_a))`. There is no RMSNorm on this branch.
Every raw Fourier vector has RMS `1/sqrt(2)`, while changing `Z` changes its
phases. The default 64 pairs give 128 features per action and a `512 → D`
projection for four actions. Negative latent values and values above 20
remain valid inputs. The full encoding is periodic with period
`2*K_Z*(latent_max - latent_min)` (2560 return units at the defaults), so it
does not uniquely represent unbounded values. The range is in the same units
as `Z`, including any reward scaling; it does not rescale rewards, flow states,
velocity outputs, or Bellman targets.

Time uses `DIFFUSION_TIME_FREQUENCIES = K` pairs and
`DIFFUSION_TIME_MIN`/`DIFFUSION_TIME_MAX`, defaulting to `0`/`1`.
An independent bias-free linear maps its `2*K` raw components into the head
width. With 64 pairs, the raw time basis has 128 components followed by a
learned `128 → D` projection. Time bounds affect only its encoding; integration
and Bellman targets still use the original `tau`. The backbone's RoPE is unchanged.

All three branches have the backbone width `D`. Context normalization gives RMS
approximately one. Independent normal weight initializations use standard
deviations `1/sqrt(A*K_Z)` and `1/sqrt(K)` for return and time projections,
respectively. These give expected branch RMS approximately one at initialization;
individual projections can differ. The branches combine once as
`h = (alpha_C*e_C + alpha_Z*e_Z + alpha_tau*e_tau)/sqrt(3)`.
The fixed, nonnegative constants are `DIFFUSION_CONTEXT_EMBEDDING_SCALE`,
`DIFFUSION_LATENT_EMBEDDING_SCALE`, and `DIFFUSION_TIME_EMBEDDING_SCALE`,
respectively. Each defaults to 1; zero disables that input branch. They multiply
the embeddings on every forward pass and are separate from Fourier
reference ranges and weight initialization. The divisor remains `sqrt(3)`;
the multipliers are not renormalized. Each scalar multiplier is divided by
`sqrt(3)` before multiplying its tensor, so a multiplier of `sqrt(3)` gives
exactly unit gain without a tensor multiply-then-divide rounding step.

`DIFFUSION_NUM_BLOCKS` residual blocks (default 2) follow the Qwen feedforward form:
`h = h + down(silu(gate(RMSNorm(h))) * up(RMSNorm(h)))`.
The gate and up projections are fused into one bias-free linear, and the down
projection is also bias-free. Backbone width `D` gives inner width `3*D` for
each gate/up branch, matching the threefold ratio in the
[Qwen3-0.6B configuration](https://huggingface.co/Qwen/Qwen3-0.6B/blob/main/config.json).
The [Qwen3 architecture](https://arxiv.org/html/2505.09388v1#S2) uses RMSNorm and
SwiGLU. A final linear readout produces all action velocities: `v = W_out h + b_out`.
Set `DIFFUSION_NUM_BLOCKS = 0` to send the summed input embeddings directly
to this linear readout. Any nonnegative integer block count is supported.
`VELOCITY_OUTPUT_INITIALIZATION_SCALE` multiplies this layer's default initial
weights and bias once at construction. The default `1.0` preserves initialization;
`0.1`, for example, starts with one tenth of those weights and biases. This is
not a forward multiplier and does not rescale trajectory or correction noise.
Input projections and residual blocks keep their existing initialization.
Head computation stays float32 under autocast, and Euler integration runs only
in the head.

[Fourier feature research](https://arxiv.org/abs/2006.10739) motivates the scalar
time representation. This is a compact architectural adaptation; those results
do not establish a benefit for this RL objective or resolve the joint-flow limits
below.

Each trajectory starts from independent Gaussian coordinates `[P,A]`:
`Z_0 = DIFFUSION_NOISE_MEAN + DIFFUSION_NOISE_STD*epsilon`,
where `epsilon ~ N(0,1)`. The fixed mean defaults to `0` and standard deviation
to `1`; both are shared across actions and states and are not learned.
They are independent of the Fourier encoding bounds and velocity initialization.
Both settings must be finite, with standard deviation nonnegative. Zero standard
deviation gives deterministic starts at the mean. Configure the head and
objective with the same mean and standard deviation. Discounts in `[0,1]`
are supported by the Gaussian Bellman correction.
The delayed encoder and head are frozen Polyak copies. `POLYAK_TAU_VALUE`
updates the whole velocity network at one rate, including input projections,
residual blocks, and final linear weights and bias. It must be finite and in
`[0,1]`: zero freezes the delayed head and one copies online parameters directly.
Frozen source parameters and buffers are unchanged. For each objective
readout, `DIFFUSION_ACTION_SAMPLES` independent full delayed trajectories are
batched together. Average their endpoint value vectors, then choose the maximum
action. One additional, independently initialized delayed trajectory supplies
its backup path and velocities. Deployment can override the sample count per
call with `q_head.q_values(z, action_samples=...)`, so online collection and
evaluation can use different counts without changing training. All modes average
endpoints before selecting an action. The default is one sample; each sample
uses all `DIFFUSION_STEPS` Euler steps.
Only online parameters are optimized.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_mask=tensors["head_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class DiffusionHead(nn.Module):
    times: torch.Tensor
    time_bands: torch.Tensor
    return_bands: torch.Tensor

    def __init__(self, *, hidden_dim, num_actions, num_steps,
                 time_frequencies, time_min, time_max,
                 latent_frequencies, latent_min, latent_max,
                 action_samples=1, velocity_output_initialization_scale=1.0,
                 noise_mean=0.0, noise_std=1.0, num_blocks=2,
                 context_embedding_scale=1.0, latent_embedding_scale=1.0, time_embedding_scale=1.0):
        super().__init__()
        if num_steps < 1:
            raise ValueError("num_steps must be positive")
        if type(num_blocks) is not int or num_blocks < 0:
            raise ValueError("num_blocks must be a nonnegative integer")
        if type(time_frequencies) is not int or time_frequencies < 1:
            raise ValueError("time_frequencies must be a positive integer")
        if type(latent_frequencies) is not int or latent_frequencies < 1:
            raise ValueError("latent_frequencies must be a positive integer")
        for name, lower, upper in (("time", time_min, time_max), ("latent", latent_min, latent_max)):
            for suffix, value in (("min", lower), ("max", upper)):
                if type(value) not in (int, float) or not -float("inf") < value < float("inf"):
                    raise ValueError(f"{name}_{suffix} must be finite")
            if not 0 < upper - lower < float("inf"):
                raise ValueError(f"{name}_max - {name}_min must be finite and positive")
        for name, value in (("context_embedding_scale", context_embedding_scale),
                            ("latent_embedding_scale", latent_embedding_scale),
                            ("time_embedding_scale", time_embedding_scale)):
            if type(value) not in (int, float) or not 0 <= value < float("inf"):
                raise ValueError(f"{name} must be finite and nonnegative")
        self.latent_min = float(latent_min)
        self.latent_max = float(latent_max)
        self.context_embedding_scale = float(context_embedding_scale)
        self.latent_embedding_scale = float(latent_embedding_scale)
        self.time_embedding_scale = float(time_embedding_scale)
        if type(action_samples) is not int or action_samples < 1:
            raise ValueError("action_samples must be a positive integer")
        if (type(velocity_output_initialization_scale) not in (int, float)
                or not 0 <= velocity_output_initialization_scale < float("inf")):
            raise ValueError("velocity_output_initialization_scale must be finite and nonnegative")
        for name, value in (("noise_mean", noise_mean), ("noise_std", noise_std)):
            if type(value) not in (int, float) or not -float("inf") < value < float("inf"):
                raise ValueError(f"{name} must be finite")
        if noise_std < 0:
            raise ValueError("noise_std must be nonnegative")
        self.noise_mean = float(noise_mean)
        self.noise_std = float(noise_std)
        self.action_samples = action_samples
        self.time_min = float(time_min)
        self.time_max = float(time_max)
        self.num_actions = num_actions
        self.width = width = hidden_dim
        self.num_steps = num_steps
        self.norm = nn.RMSNorm(
            hidden_dim, eps=1e-5, elementwise_affine=False, device="cpu", dtype=torch.float32,
        )
        self.noisy_return = nn.Linear(
            2 * latent_frequencies * num_actions, width, bias=False, device="cpu", dtype=torch.float32,
        )
        self.register_buffer(
            "return_bands", torch.pi * torch.arange(1, latent_frequencies + 1, dtype=torch.float32) / latent_frequencies,
            persistent=False,
        )
        self.register_buffer(
            "time_bands", torch.pi * torch.arange(1, time_frequencies + 1, dtype=torch.float32) / time_frequencies,
            persistent=False,
        )
        self.time_embedding = nn.Linear(2 * time_frequencies, width, bias=False, device="cpu", dtype=torch.float32)
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.RMSNorm(width, eps=1e-6), nn.Linear(width, 6 * width, bias=False),
                SwiGLU(), nn.Linear(3 * width, width, bias=False),
            ) for _ in range(num_blocks)
        ])
        self.output = nn.Linear(width, num_actions)
        with torch.no_grad():
            self.output.weight.mul_(velocity_output_initialization_scale)
            self.output.bias.mul_(velocity_output_initialization_scale)
            # Match expected branch RMS at initialization, before the shared sum.
            nn.init.normal_(self.noisy_return.weight, std=(num_actions * latent_frequencies)**-0.5)
            nn.init.normal_(self.time_embedding.weight, std=(1 / time_frequencies)**0.5)
        self.register_buffer("times", torch.arange(num_steps).float() / num_steps)

    def forward(self, z, samples, *, times):
        """Final features [P,H], noisy returns [P,T,A], times [T] -> velocities."""
        with torch.autocast(device_type=z.device.type, enabled=False):
            context = self.norm(z.float())
            # Bounds set the reference range; noisy values outside it are not clipped.
            normalized_returns = (samples.float() - self.latent_min) / (self.latent_max - self.latent_min)
            return_angles = normalized_returns[..., None] * self.return_bands
            return_features = torch.cat((return_angles.sin(), return_angles.cos()), dim=-1).flatten(-2)
            noisy = self.noisy_return(return_features)
            normalized_time = (times.float() - self.time_min) / (self.time_max - self.time_min)
            angles = normalized_time[:, None] * self.time_bands
            time_features = torch.cat((angles.sin(), angles.cos()), dim=-1)
            time = self.time_embedding(time_features)
            # Combine scalar factors first so sqrt(3) gives exactly unit gain.
            hidden = ((self.context_embedding_scale / 3**0.5) * context[:, None]
                      + (self.latent_embedding_scale / 3**0.5) * noisy
                      + (self.time_embedding_scale / 3**0.5) * time[None])
            for block in self.blocks:
                hidden = hidden + block(hidden)
            return self.output(hidden)

    @torch.no_grad()
    def trajectory(self, z, *, initial_noise):
        """Evolve already-transformed starting samples initial_noise of shape [P,A]."""
        if initial_noise.shape != (z.shape[0], self.num_actions):
            raise ValueError("initial_noise must have shape [P, A]")
        x = initial_noise
        samples, velocities = [], []
        for k in range(self.num_steps):
            v = self(z, x[:, None, :], times=self.times[k:k + 1])[:, 0]
            samples.append(x)
            velocities.append(v)
            x = x + v / self.num_steps
        return torch.stack(samples, 1), torch.stack(velocities, 1), x

    @torch.no_grad()
    def q_values(self, z, *, action_samples=None, generator=None):
        """Mean trajectory endpoints; optionally override the count for this call."""
        count = self.action_samples if action_samples is None else action_samples
        if type(count) is not int or count < 1:
            raise ValueError("action_samples must be a positive integer")
        leading_shape = z.shape[:-1]
        features = z.reshape(-1, z.shape[-1])
        rows = features.shape[0]
        if count > 1:
            features = features.repeat_interleave(count, dim=0)
        noise = torch.randn(features.shape[0], self.num_actions, device=z.device,
                            dtype=torch.float32, generator=generator)
        noise = self.noise_mean + self.noise_std * noise
        _, _, returns = self.trajectory(features, initial_noise=noise)
        if count > 1:
            returns = returns.reshape(rows, count, self.num_actions).mean(dim=1)
        return returns.reshape(*leading_shape, self.num_actions)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)

q_head = (
    DiffusionHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_blocks=DIFFUSION_NUM_BLOCKS,
        velocity_output_initialization_scale=VELOCITY_OUTPUT_INITIALIZATION_SCALE,
        num_steps=DIFFUSION_STEPS,
        action_samples=DIFFUSION_ACTION_SAMPLES,
        noise_mean=DIFFUSION_NOISE_MEAN,
        noise_std=DIFFUSION_NOISE_STD,
        time_frequencies=DIFFUSION_TIME_FREQUENCIES,
        time_min=DIFFUSION_TIME_MIN,
        time_max=DIFFUSION_TIME_MAX,
        latent_frequencies=DIFFUSION_LATENT_FREQUENCIES,
        latent_min=DIFFUSION_LATENT_MIN,
        latent_max=DIFFUSION_LATENT_MAX,
        context_embedding_scale=DIFFUSION_CONTEXT_EMBEDDING_SCALE,
        latent_embedding_scale=DIFFUSION_LATENT_EMBEDDING_SCALE,
        time_embedding_scale=DIFFUSION_TIME_EMBEDDING_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()

print(encoder)
print(q_head)

## Train

For each readout, draw `epsilon ~ N(0,1)` independently of every selector
and backup starting sample, and hold it fixed across diffusion times `tau`.
For configured mean `mu` and standard deviation `sigma`, construct:

- `correction = (1-gamma)*mu + sqrt(1-gamma**2)*sigma*epsilon`
- `z = tau*r + gamma*z_next + (1-tau)*correction`
- `target = r + gamma*v_next - correction`

The supplied discount `gamma` includes termination and must be in `[0,1]`.
At discount one, the correction is zero. At outer termination it is a fresh
`N(mu, sigma**2)` draw, so no terminal readout is needed. At `tau=0`, the
Bellman-transformed coordinate has mean `mu` and variance `sigma**2` for
any supported discount. Rewards keep example 2's scale; the paper's optional
reward normalization is not applied.

As in example 2, inner episode endings keep the return connected across repeats.
The existing one-step DQN alignment carries automatic reset rows to the next
decision, combining their rewards and discounts. The correction uses the
combined discount. It never continues through another executed decision or
sample boundary.

Start the online training input from the delayed **current-state** action vector,
then replace only the taken action's coordinate with `x`. Its target velocity is
`target`. The head reads the entire vector and predicts all action velocities,
but the loss selects only the action taken. Other output coordinates receive no
direct loss; shared head parameters still train through the selected output.

This is an experimental joint-head extension of the paper's scalar Q-only loss.
The paper's flow-matching argument requires the conditional velocity of a specified
path. Here each taken action defines a different vector input distribution and
supervises only one velocity coordinate. Matching these partial targets does not
establish transport of any one joint path, or Bellman-consistent generated
marginals. Replay also does not identify the true joint distribution of returns
for actions that were not taken. These are limits of the derivation, not evidence
that the coupled head cannot work.

The selector trajectories, single backup trajectory, and correction noise use
independent draws. Average the `DIFFUSION_ACTION_SAMPLES` selector endpoints,
then take the maximum action and keep it fixed across every diffusion time of
the backup trajectory. `prepare` builds detached inputs and targets; the online
head then predicts velocities. Loss averages only taken-action errors over
valid readouts and **all** `DIFFUSION_STEPS`. Deployment uses the same endpoint
averaging before `argmax`, with an independently selectable sample count for
each call. The sample count changes the action estimate, not
the number of Euler steps per trajectory or optimizer updates.


In [ ]:
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": q_head.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objective = DiffusionDqnObjective(
    reward=None,
    noise_mean=DIFFUSION_NOISE_MEAN,
    noise_std=DIFFUSION_NOISE_STD,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    if type(POLYAK_TAU_VALUE) not in (int, float) or not 0 <= POLYAK_TAU_VALUE <= 1:
        raise ValueError("POLYAK_TAU_VALUE must be finite and in [0, 1]")
    modules.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        sequence_ids = objective_data["sequence_ids"]
        out = encoder(inputs)
        with torch.no_grad():
            delayed_out = delayed_encoder(inputs)
            features = delayed_out["z"]
            # Average full selector endpoints, then keep the chosen action for every tau.
            action_values = backup_value_head.q_values(features)
            # Independent full path supplies the Bellman samples and velocities.
            initial_noise = torch.randn(features.shape[0], q_head.num_actions, device=device)
            initial_noise = (backup_value_head.noise_mean
                             + backup_value_head.noise_std * initial_noise)
            samples, velocities, _ = backup_value_head.trajectory(
                features, initial_noise=initial_noise,
            )
            flow_inputs, targets, actions, weight = objective.prepare(
                objective_data=objective_data,
                sequence_ids=sequence_ids,
                step_ids=objective_data["step_ids"],
                backup_samples=samples,
                backup_value_predictions=velocities,
                backup_action_predictions=action_values,
                times=q_head.times,
                noise=torch.randn(features.shape[0], device=device),
            )
        predictions = q_head(out["z"], flow_inputs, times=q_head.times)
        losses, metrics = objective(
            predictions=predictions, targets=targets, actions=actions, weight=weight,
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak_update(
            target=delayed_encoder.embedding,
            source=encoder.embedding,
            tau=POLYAK_TAU_EMBEDDING,
        )
        polyak_update(
            target=delayed_encoder.backbone,
            source=encoder.backbone,
            tau=POLYAK_TAU_BACKBONE,
        )
        polyak_update(
            target=backup_value_head,
            source=q_head,
            tau=POLYAK_TAU_VALUE,
        )
    assert loss is not None
    return (loss, metrics)

## Run

Each cycle calls `run_train(num_steps=TRAIN_STEPS)`. Training chooses backup
actions using `DIFFUSION_ACTION_SAMPLES` full trajectory endpoints. For inference,
pass the encoder's final features to `q_head.q_values(z, action_samples=count)`
and choose the highest-scoring action. For example, an online collector could
use `action_samples=2` and evaluation could use `action_samples=8`; these overrides
apply only to their calls and leave the training count unchanged. Omitting the
argument uses the head's configured count. Notebook 15's scalar head must be
replaced with this notebook's `DiffusionHead` when loading this checkpoint.
Reconstruct the head with the same Gaussian noise mean and standard deviation;
these fixed construction settings are not stored in the weights file.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(num_steps=TRAIN_STEPS)
    print(
        f"cycle={cycle} train  loss={loss.item():.4f}  "
        f"diffusion_steps={torch.as_tensor(metrics['per_diffusion_step']).tolist()}"
    )

loader.close()

## Save Locally

Save the encoder, diffusion head, and unchanged tokenizer. Reconstruct the same
components and settings before loading. Publishing is a separate reviewed step.


In [ ]:
modules.eval().to("cpu")
save_model(model=modules, path=OUTPUT)
save_tokenizer(tokenizer=tokenizer, path=OUTPUT)
print(f"Saved encoder, diffusion head, and tokenizer to {OUTPUT}")
